In [ ]:
from abc import ABC, abstractmethod
from collections import Counter
from typing import Protocol

import tiktoken


In [2]:
class TokenizerProtocol(Protocol):
    def encode(self, text: str) -> list[int]: ...
    def decode(self, tokens: list[int]) -> str: ...

In [3]:
class Tokenizer(ABC):
    def __init__(
        self,
        bos_str: str = "<BOS>",
        eos_str: str = "<EOS>",
        bos_id: int = 0,
        eos_id: int = 1,
    ):
        self._bos_str = bos_str
        self._eos_str = eos_str

        self._bos_id = bos_id
        self._eos_id = eos_id

        self._piece_to_token: dict[str, int] = {
            bos_str: bos_id,
            eos_str: eos_id,
        }

        self._token_to_piece: dict[int, str] = {
            bos_id: bos_str,
            eos_id: eos_str,
        }

    @abstractmethod
    def encode(self, text: str) -> list[int]:
        pass

    @abstractmethod
    def decode(self, tokens: list[int]) -> str:
        pass

    @property
    def vocab_size(self):
        return len(self._piece_to_token)

In [4]:
class BPETokenizer(Tokenizer):
    def __init__(self, vocab_size: int = 100, **kwargs):
        super().__init__(**kwargs)

        self._target_vocab_size = vocab_size
        self._merges: dict[tuple[str, str], int] = {}

    def _get_pairs(self, symbols: list[str]) -> set[tuple[str, str]]:

        pairs = set()

        for i in range(len(symbols) - 1):
            pairs.add((symbols[i], symbols[i + 1]))

        return pairs

    def _get_most_frequent_pair(self, words: list[list[str]]):
        counter = Counter()
        for word in words:
            pairs = self._get_pairs(word)

            for pair in pairs:
                counter[pair] += 1

        if not counter:
            return None

        return counter.most_common(1)[0][0]

    def _merge_pair(self, symbols: list[str], pair: tuple[str, str]):
        result = []
        i = 0
        while i < len(symbols):
            if (
                i < len(symbols) - 1
                and symbols[i] == pair[0]
                and symbols[i + 1] == pair[1]
            ):
                result.append(symbols[i] + symbols[i + 1])
                i += 2
            else:
                result.append(symbols[i])
                i += 1

        return result

    def train(self, texts: list[str]):
        words = []

        for text in texts:
            for word in text.split():
                symbols = list(word)

                words.append(symbols)

                for symbol in symbols:
                    if symbol not in self._piece_to_token:
                        token_id = len(self._piece_to_token)

                        self._piece_to_token[symbol] = token_id

                        self._token_to_piece[token_id] = symbol

        while self.vocab_size < self._target_vocab_size:
            pair = self._get_most_frequent_pair(words)
            if pair is None:
                break
            rank = len(self._merges)
            self._merges[pair] = rank
            new_token = "".join(pair)

            if new_token not in self._piece_to_token:
                token_id = len(self._piece_to_token)

                self._piece_to_token[new_token] = token_id

                self._token_to_piece[token_id] = new_token
            for i, word in enumerate(words):
                words[i] = self._merge_pair(word, pair)

    def _apply_bpe(self, symbols: list[str]):
        while True:
            pairs = self._get_pairs(symbols)
            if not pairs:
                break

            available = [pair for pair in pairs if pair in self._merges]
            if not available:
                break

            best_pair = min(available, key=lambda x: self._merges[x])
            symbols = self._merge_pair(symbols, best_pair)

        return symbols

    def encode(self, text: str) -> list[int]:
        tokens = [self._bos_id]
        for word in text.split():
            symbols = list(word)
            symbols = self._apply_bpe(symbols)
            for symbol in symbols:
                tokens.append(self._piece_to_token[symbol])
        tokens.append(self._eos_id)

        return tokens

    def decode(self, tokens: list[int]) -> str:
        pieces = []
        for token in tokens:
            if token in (self._bos_id, self._eos_id):
                continue
            pieces.append(self._token_to_piece[token])

        return " ".join(pieces)

In [5]:
text = [
    """
Я памятник себе воздвиг нерукотворный,
К нему не зарастет народная тропа,
Вознесся выше он главою непокорной
Александрийского столпа.

Нет, весь я не умру — душа в заветной лире
Мой прах переживет и тленья убежит —
И славен буду я, доколь в подлунном мире
Жив будет хоть один пиит.

Слух обо мне пройдет по всей Руси великой,
И назовет меня всяк сущий в ней язык,
И гордый внук славян, и финн, и ныне дикой
Тунгус, и друг степей калмык.

И долго буду тем любезен я народу,
Что чувства добрые я лирой пробуждал,
Что в мой жестокий век восславил я Свободу
И милость к падшим призывал.

Веленью божию, о муза, будь послушна,
Обиды не страшась, не требуя венца,
Хвалу и клевету приемли равнодушно
И не оспоривай глупца.
""",
    """
    О чем шумите вы, народные витии?
Зачем анафемой грозите вы России?
Что возмутило вас? волнения Литвы?
Оставьте: это спор славян между собою,
Домашний, старый спор, уж взвешенный судьбою,
Вопрос, которого не разрешите вы.
 Уже давно между собою
 Враждуют эти племена;
 Не раз клонилась под грозою
 То их, то наша сторона.
 Кто устоит в неравном споре:
 Кичливый лях, иль верный росс?
Славянские ль ручьи сольются в русском море?
 Оно ль иссякнет? вот вопрос.
 Оставьте нас: вы не читали
 Сии кровавые скрижали;
 Вам непонятна, вам чужда
 Сия семейная вражда;
 Для вас безмолвны Кремль и Прага;
 Бессмысленно прельщает вас
 Борьбы отчаянной отвага —
 И ненавидите вы нас...
 За что ж? ответствуйте: за то ли,
Что на развалинах пылающей Москвы
 Мы не признали наглой воли
 Того, под кем дрожали вы?
 За то ль, что в бездну повалили
Мы тяготеющий над царствами кумир
 И нашей кровью искупили
 Европы вольность, честь и мир?..
Вы грозны на словах — попробуйте на деле!
Иль старый богатырь, покойный на постеле,
Не в силах завинтить свой измаильский штык?
Иль русского царя уже бессильно слово?
 Иль нам с Европой спорить ново?
 Иль русский от побед отвык?
Иль мало нас? Или от Перми до Тавриды,
От финских хладных скал до пламенной Колхиды,
 От потрясенного Кремля
 До стен недвижного Китая,
 Стальной щетиною сверкая,
 Не встанет русская земля?..
 Так высылайте ж к нам, витии,
 Своих озлобленных сынов:
 Есть место им в полях России,
 Среди нечуждых им гробов.
""",
    """
Не мысля гордый свет забавить,
Вниманье дружбы возлюбя,
Хотел бы я тебе представить
Залог достойнее тебя,
Достойнее души прекрасной,
Святой исполненной мечты,
Поэзии живой и ясной,
Высоких дум и простоты;
Но так и быть — рукой пристрастной
Прими собранье пестрых глав,
Полусмешных, полупечальных,
Простонародных, идеальных,
Небрежный плод моих забав,
Бессонниц, легких вдохновений,
Незрелых и увядших лет,
Ума холодных наблюдений
И сердца горестных замет.
""",
    """
На берегу пустынных волн
Стоял он, дум великих полн,
И вдаль глядел. Пред ним широко
Река неслася; бедный чёлн
По ней стремился одиноко.
По мшистым, топким берегам
Чернели избы здесь и там,
Приют убогого чухонца;
И лес, неведомый лучам
В тумане спрятанного солнца,
Кругом шумел.
И думал он:
Отсель грозить мы будем шведу,
Здесь будет город заложен
На зло надменному соседу.
Природой здесь нам суждено
В Европу прорубить окно,1
Ногою твердой стать при море.
Сюда по новым им волнам
Все флаги в гости будут к нам,
И запируем на просторе.
Прошло сто лет, и юный град,
Полнощных стран краса и диво,
Из тьмы лесов, из топи блат
Вознесся пышно, горделиво;
Где прежде финский рыболов,
Печальный пасынок природы,
Один у низких берегов
Бросал в неведомые воды
Свой ветхой невод, ныне там
По оживленным берегам
Громады стройные теснятся
Дворцов и башен; корабли
Толпой со всех концов земли
К богатым пристаням стремятся;
В гранит оделася Нева;
Мосты повисли над водами;
Темно-зелеными садами
Ее покрылись острова,
И перед младшею столицей
Померкла старая Москва,
Как перед новою царицей
Порфироносная вдова.
Люблю тебя, Петра творенье,
Люблю твой строгий, стройный вид,
Невы державное теченье,
Береговой ее гранит,
Твоих оград узор чугунный,
Твоих задумчивых ночей
Прозрачный сумрак, блеск безлунный,
Когда я в комнате моей
Пишу, читаю без лампады,
И ясны спящие громады
Пустынных улиц, и светла
Адмиралтейская игла,
И, не пуская тьму ночную
На золотые небеса,
Одна заря сменить другую
Спешит, дав ночи полчаса2.
Люблю зимы твоей жестокой
Недвижный воздух и мороз,
Бег санок вдоль Невы широкой,
Девичьи лица ярче роз,
И блеск, и шум, и говор балов,
А в час пирушки холостой
Шипенье пенистых бокалов
И пунша пламень голубой.
Люблю воинственную живость
Потешных Марсовых полей,
Пехотных ратей и коней
Однообразную красивость,
В их стройно зыблемом строю
Лоскутья сих знамен победных,
Сиянье шапок этих медных,
На сквозь простреленных в бою.
Люблю, военная столица,
Твоей твердыни дым и гром,
Когда полнощная царица
Дарует сына в царской дом,
Или победу над врагом
Россия снова торжествует,
Или, взломав свой синий лед,
Нева к морям его несет
И, чуя вешни дни, ликует.
Красуйся, град Петров, и стой
Неколебимо как Россия,
Да умирится же с тобой
И побежденная стихия;
Вражду и плен старинный свой
Пусть волны финские забудут
И тщетной злобою не будут
Тревожить вечный сон Петра!
Была ужасная пора,
Об ней свежо воспоминанье...
Об ней, друзья мои, для вас
Начну свое повествованье.
Печален будет мой рассказ.

""",
]

In [6]:
tokenizer = BPETokenizer(vocab_size=30)
tokenizer.train(text)
print(tokenizer._merges)
ids = tokenizer.encode("")
print(ids)
print(tokenizer.decode(ids))

{}
[0, 1]

